In [4]:
"""
Elasticidade-Preço por Dia da Semana — Venda B2B de Carne In Natura
=====================================================================
(Versão usando statsmodels)

Pergunta: a elasticidade-preço da demanda é diferente entre os dias
da semana, ou é razoável assumir uma elasticidade única?

Dados esperados (formato longo, uma linha por observação):
    dia_semana | preco | quantidade
    Segunda    | 12.50 | 480
    Segunda    | 13.00 | 460
    ...
    Sexta      | 11.80 | 510

Com 13 observações por dia x 7 dias = 91 linhas no total.

Abordagem:
  1. Modelo RESTRITO : elasticidade única, intercepto varia por dia
       ln(Q) = β0 + Σ γ_i·Dia_i + β·ln(P) + ε
  2. Modelo COMPLETO : elasticidade E intercepto variam por dia
       ln(Q) = β0 + Σ γ_i·Dia_i + β·ln(P) + Σ δ_i·(Dia_i × ln(P)) + ε

  O teste de Chow (F-test, via anova_lm) compara os dois modelos: se
  os termos de interação (δ_i) forem conjuntamente significativos, há
  evidência estatística de que a elasticidade difere entre os dias.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# -------------------------------------------------------------------
# 1. CARREGAR OS DADOS
# -------------------------------------------------------------------

df = pd.read_excel('RPVMM.xlsx','Planilha6' )

n_total = len(df)
n_dias = df["Dia_da_Semana"].nunique()
print(f"Total de observações: {n_total} ({n_dias} dias x "
      f"{n_total // n_dias} obs/dia)\n")

# -------------------------------------------------------------------
# 2. TRANSFORMAÇÃO LOG-LOG
# -------------------------------------------------------------------
df["ln_preco"] = np.log(df["Preço"])
df["ln_qtd"] = np.log(df["Volume_Realizado"])

# Fixamos a categoria de referência explicitamente (opcional, mas deixa
# o resultado mais previsível do que depender da ordem alfabética padrão)
dia_referencia = "4"
#df["Dia_da_Semana"] = pd.Categorical(
#    df["Dia_da_Semana"], categories=[dia_referencia] + [d for d in dias if d != dia_referencia]
#)

# -------------------------------------------------------------------
# 3. MODELO RESTRITO — elasticidade única, intercepto varia por dia
# -------------------------------------------------------------------
modelo_restrito = smf.ols(
    "ln_qtd ~ C(Dia_da_Semana) + ln_preco", data=df
).fit()

print("=" * 65)
print("MODELO RESTRITO (elasticidade única para todos os dias)")
print("=" * 65)
print(modelo_restrito.summary())

beta_unico = modelo_restrito.params["ln_preco"]
print(f"\nElasticidade comum estimada: {beta_unico:.4f}")

# -------------------------------------------------------------------
# 4. MODELO COMPLETO — elasticidade E intercepto variam por dia
#    O "*" no formula gera automaticamente os termos principais e as
#    interações: C(dia_semana) + ln_preco + C(dia_semana):ln_preco
# -------------------------------------------------------------------
modelo_completo = smf.ols(
    "ln_qtd ~ C(Dia_da_Semana) * ln_preco", data=df
).fit()

print("\n" + "=" * 65)
print("MODELO COMPLETO (elasticidade específica por dia)")
print("=" * 65)
print(modelo_completo.summary())

beta_referencia = modelo_completo.params["ln_preco"]
print(f"\nElasticidade do dia de referência ({dia_referencia}): {beta_referencia:.4f}")
print("\nElasticidade por dia (referência + desvio estimado):")
print(f"  {dia_referencia:<10s}: {beta_referencia:.4f}   (baseline)")

for nome_param in modelo_completo.params.index:
    if "C(Dia da Semana)" in nome_param and ":ln_preco" in nome_param:
        # nome_param tem o formato "C(dia_semana)[T.Sexta]:ln_preco"
        dia_nome = nome_param.split("[T.")[1].split("]")[0]
        delta = modelo_completo.params[nome_param]
        elasticidade_dia = beta_referencia + delta
        print(f"  {dia_nome:<10s}: {elasticidade_dia:.4f}   (desvio: {delta:+.4f}, "
              f"p-valor do desvio: {modelo_completo.pvalues[nome_param]:.4g})")

# -------------------------------------------------------------------
# 5. TESTE DE CHOW (F-TEST) — as elasticidades diferem de fato?
#    H0: todos os δ_i = 0 (elasticidade única é suficiente)
#    H1: pelo menos um δ_i ≠ 0 (elasticidades diferem por dia)
#    anova_lm compara os dois modelos aninhados diretamente.
# -------------------------------------------------------------------
tabela_anova = anova_lm(modelo_restrito, modelo_completo)

print("\n" + "=" * 65)
print("TESTE DE CHOW (a elasticidade difere entre os dias?)")
print("=" * 65)
print(tabela_anova)

F_stat = tabela_anova["F"].iloc[1]
p_valor_chow = tabela_anova["Pr(>F)"].iloc[1]

alpha = 0.05
print(f"\nEstatística F = {F_stat:.4f}, p-valor = {p_valor_chow:.4g}")
if p_valor_chow < alpha:
    print(f">> p-valor < {alpha}: rejeita-se H0. Há evidência estatística de "
          "que a elasticidade-preço DIFERE entre os dias da semana. "
          "Reportar elasticidades separadas (modelo completo) é justificado.")
else:
    print(f">> p-valor >= {alpha}: não se rejeita H0. Não há evidência "
          "estatística suficiente de que as elasticidades diferem. "
          "O modelo com elasticidade única (mais simples e parcimonioso) "
          "é preferível.")

# -------------------------------------------------------------------
# 5b. DIAGNÓSTICO: VALE A PENA TROCAR PARA HUBER?
#    Checamos resíduos studentizados do modelo completo (OLS) por dia.
#    Se houver outliers concentrados em dias específicos, o Huber pode
#    ajudar. Caso contrário, fique com OLS (o teste de Chow acima já é
#    válido e bem estabelecido).
# -------------------------------------------------------------------
influencia = modelo_completo.get_influence()
residuos_estudentizados = influencia.resid_studentized_internal
df["residuo_estudentizado"] = residuos_estudentizados

print("\n" + "=" * 65)
print("DIAGNÓSTICO: RESÍDUOS ESTUDENTIZADOS POR DIA (modelo completo OLS)")
print("=" * 65)
resumo_residuos = df.groupby("Dia_da_Semana", observed=True)["residuo_estudentizado"].agg(
    ["mean", "std", "max", "min"]
)
print(resumo_residuos)
print("\nResíduo estudentizado > |3| costuma indicar outlier. Se estiverem "
      "concentrados em 1-2 dias, considere investigar antes de trocar de "
      "estimador; se espalhados e moderados, o OLS provavelmente já basta.")

# -------------------------------------------------------------------
# 5c. SE OPTAR POR HUBER: aplicar aos DOIS modelos (restrito e completo)
#    e testar a diferença de elasticidades com um teste de WALD, não
#    com anova_lm (que exige RSS de OLS e não se aplica ao M-estimador
#    do Huber).
# -------------------------------------------------------------------
print("\n" + "=" * 65)
print("VERSÃO ROBUSTA (Huber via RLM) — só usar se o diagnóstico acima "
      "indicar outliers relevantes")
print("=" * 65)

import patsy

y_patsy, X_restrito_patsy = patsy.dmatrices(
    "ln_qtd ~ C(Dia_da_Semana) + ln_preco", data=df, return_type="dataframe"
)
_, X_completo_patsy = patsy.dmatrices(
    "ln_qtd ~ C(Dia_da_Semana) * ln_preco", data=df, return_type="dataframe"
)

modelo_restrito_huber = sm.RLM(
    y_patsy, X_restrito_patsy, M=sm.robust.norms.HuberT()
).fit()
modelo_completo_huber = sm.RLM(
    y_patsy, X_completo_patsy, M=sm.robust.norms.HuberT()
).fit()

beta_unico_huber = modelo_restrito_huber.params["ln_preco"]
beta_referencia_huber = modelo_completo_huber.params["ln_preco"]

print(f"Elasticidade única (Huber, restrito):        {beta_unico_huber:.4f}"
      f"  [OLS: {beta_unico:.4f}]")
print(f"Elasticidade dia referência (Huber, completo): {beta_referencia_huber:.4f}"
      f"  [OLS: {beta_referencia:.4f}]")

# Teste de Wald conjunto: todos os coeficientes de interação (Dia_i:ln_preco) = 0
# Construímos a matriz de restrição R manualmente (mais seguro do que uma
# string de fórmula, já que os nomes das colunas do patsy contêm colchetes)
nomes_interacao = [c for c in X_completo_patsy.columns if ":ln_preco" in c]
n_params = len(modelo_completo_huber.params)
R = np.zeros((len(nomes_interacao), n_params))
for i, nome in enumerate(nomes_interacao):
    idx_param = list(modelo_completo_huber.params.index).index(nome)
    R[i, idx_param] = 1.0

teste_wald = modelo_completo_huber.wald_test(R)

print(f"\nTeste de Wald (H0: todas as interações Dia×ln(P) = 0, versão Huber):")
print(teste_wald)
print("\nCompare a conclusão deste teste com a do teste de Chow (OLS) acima. "
      "Se ambos apontarem na mesma direção, a conclusão é robusta ao "
      "método de estimação. Se divergirem, os outliers identificados no "
      "diagnóstico 5b provavelmente estão distorcendo o resultado do OLS.")

# -------------------------------------------------------------------
# 6. VALIDAÇÃO CRUZADA (overfitting) — comparando os dois modelos
#    statsmodels não tem CV nativo, então implementamos manualmente
#    com KFold do sklearn, mas usando o modelo (fórmula) do statsmodels
#    em cada fold para manter a mesma especificação.
# -------------------------------------------------------------------
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=7, shuffle=True, random_state=42)


def mae_cv_formula(formula, df, kf):
    erros = []
    for idx_treino, idx_teste in kf.split(df):
        df_treino = df.iloc[idx_treino]
        df_teste = df.iloc[idx_teste]
        modelo = smf.ols(formula, data=df_treino).fit()
        y_pred = modelo.predict(df_teste)
        erros.extend(np.abs(df_teste["ln_qtd"].values - y_pred.values))
    return np.mean(erros)


mae_cv_restrito = mae_cv_formula("ln_qtd ~ C(Dia_da_Semana) + ln_preco", df, kf)
mae_cv_completo = mae_cv_formula("ln_qtd ~ C(Dia_da_Semana) * ln_preco", df, kf)
mae_in_restrito = mean_absolute_error(df["ln_qtd"], modelo_restrito.fittedvalues)
mae_in_completo = mean_absolute_error(df["ln_qtd"], modelo_completo.fittedvalues)

print("\n" + "=" * 65)
print("OVERFITTING: MAE in-sample vs. 7-fold CV")
print("=" * 65)
print(f"{'Modelo':<20}{'MAE in-sample':>16}{'MAE CV':>12}{'Diferença':>14}")
print(f"{'Restrito (único)':<20}{mae_in_restrito:>16.4f}{mae_cv_restrito:>12.4f}"
      f"{mae_cv_restrito - mae_in_restrito:>14.4f}")
print(f"{'Completo (por dia)':<20}{mae_in_completo:>16.4f}{mae_cv_completo:>12.4f}"
      f"{mae_cv_completo - mae_in_completo:>14.4f}")
print("\nSe o modelo completo tiver um salto muito maior de in-sample "
      "para CV (comparado ao restrito), é sinal de que os parâmetros "
      "extras de elasticidade estão ajustando ruído, não sinal real — "
      "mesmo que o teste de Chow tenha dado significativo.")

# -------------------------------------------------------------------
# 7. RESUMO FINAL
# -------------------------------------------------------------------
print("\n" + "=" * 65)
print("RESUMO FINAL")
print("=" * 65)
print(f"Teste de Chow: F={F_stat:.4f}, p={p_valor_chow:.4g}")
print(f"R² ajustado (restrito):  {modelo_restrito.rsquared_adj:.4f}")
print(f"R² ajustado (completo):  {modelo_completo.rsquared_adj:.4f}")
print(f"AIC (restrito):          {modelo_restrito.aic:.2f}")
print(f"AIC (completo):          {modelo_completo.aic:.2f}  "
      f"(menor AIC = melhor trade-off ajuste/complexidade)")
print(f"Decisão sugerida: {'usar elasticidades por dia' if p_valor_chow < alpha else 'usar elasticidade única'}")

Total de observações: 76 (7 dias x 10 obs/dia)

MODELO RESTRITO (elasticidade única para todos os dias)
                            OLS Regression Results                            
Dep. Variable:                 ln_qtd   R-squared:                       0.835
Model:                            OLS   Adj. R-squared:                  0.818
Method:                 Least Squares   F-statistic:                     49.12
Date:                Sat, 12 Sep 2026   Prob (F-statistic):           3.77e-24
Time:                        15:15:07   Log-Likelihood:                -53.059
No. Observations:                  76   AIC:                             122.1
Df Residuals:                      68   BIC:                             140.8
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
----------------